# Sesión 3 — Autocorrelación, Calidad de Datos y Modelos Baseline

**Versión reorganizada del notebook del taller**

Este notebook conserva el objetivo central del taller: construir una serie temporal semanal de casos de dengue, revisar su calidad, estudiar autocorrelación y estacionariedad, entrenar modelos baseline y evaluar sus residuos.

La diferencia principal frente a la versión original es el orden metodológico:

1. Construcción de la serie temporal.
2. Diagnóstico de calidad de datos.
3. División temporal `train/test`.
4. Imputación usando solo información del entrenamiento.
5. Modelos baseline sobre la escala original.
6. Evaluación y diagnóstico de residuos.
7. Autocorrelación, estacionariedad y transformaciones.
8. Validación cruzada temporal.

La idea es evitar fuga de información, mantener una comparación justa y dejar el notebook más fácil de mantener mediante funciones reutilizables.

---

## 1. Configuración del entorno

En esta sección se instalan y cargan las librerías necesarias.  
El notebook usa `statsforecast` para los modelos baseline, `utilsforecast` para métricas y `statsmodels` para pruebas estadísticas de series temporales.

In [33]:
# Si estás en Google Colab, ejecuta esta celda.
# En un entorno local donde ya tengas las librerías instaladas, puedes omitirla.

!pip install statsforecast utilsforecast statsmodels -q

In [34]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from statsmodels.tsa.stattools import acf, adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, WindowAverage, RandomWalkWithDrift

from utilsforecast.losses import mae, rmse, smape
from utilsforecast.evaluation import evaluate

warnings.filterwarnings("ignore")

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


---

## 2. Parámetros generales del análisis

Aquí se definen las variables principales del proyecto.  
Si el archivo cambia de nombre, normalmente solo necesitas modificar `DATA_PATH`.

La serie se trabajará en formato compatible con Nixtla:

```text
unique_id | ds | y
```

donde:

- `unique_id`: identificador de la serie.
- `ds`: fecha semanal.
- `y`: número de casos de dengue en esa semana.

In [ ]:
# ============================================================
# Parámetros principales
# ============================================================

DATA_PATH = "/content/datos_dengue_202604282143.csv"

DATE_COLUMN = "fec_not"
SERIES_ID = "dengue_cali"

FREQ = "W-MON"          # Serie semanal anclada a lunes
SEASON_LENGTH = 52*4      # Estacionalidad anual aproximada en semanas

TRAIN_FRACTION = 0.70   # División temporal 70/30 si no se usa fecha de corte
CUTOFF_DATE = "2020-01-01"      # Ejemplo: "2020-01-01". Si queda None, se usa TRAIN_FRACTION.

RANDOM_SEED = 42

print("Parámetros definidos")

Parámetros definidos


---

## 3. Funciones auxiliares del flujo completo

Para mantener el notebook simple y ordenado, primero se definen funciones pequeñas y reutilizables.

Estas funciones cubren:

- carga y preparación de datos;
- construcción de la serie semanal;
- diagnóstico de faltantes;
- imputación estacional sin usar información futura;
- gráficos de serie temporal;
- entrenamiento y evaluación de baselines;
- pruebas de autocorrelación y estacionariedad.

In [36]:
# ============================================================
# 3.1 Carga y construcción de la serie temporal
# ============================================================

def find_data_path(data_path: str) -> Path:
    """
    Busca el archivo de datos principal.
    Primero intenta usar DATA_PATH. Si no existe, busca archivos de dengue
    en /content y /mnt/data.
    """
    path = Path(data_path)

    if path.exists():
        return path

    candidates = (
        list(Path("/content").glob("*dengue*.csv"))
        + list(Path("/mnt/data").glob("*dengue*.csv"))
        + list(Path(".").glob("*dengue*.csv"))
    )

    if not candidates:
        raise FileNotFoundError(
            "No se encontró el archivo CSV. Actualiza DATA_PATH o sube el archivo al entorno."
        )

    print(f"⚠️ DATA_PATH no existe. Se usará el archivo encontrado: {candidates[0]}")
    return candidates[0]


def load_csv_data(data_path: str) -> pd.DataFrame:
    """
    Carga el CSV usando una ruta flexible.
    """
    path = find_data_path(data_path)
    df_raw = pd.read_csv(path)
    print(f"Archivo cargado: {path}")
    print(f"Dimensiones: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
    return df_raw


def monday_of_week(date_series: pd.Series) -> pd.Series:
    """
    Convierte cualquier fecha al lunes de su semana.
    Esto evita problemas de parseo con strings tipo '2009-W53'.
    """
    dates = pd.to_datetime(date_series, errors="coerce")
    return dates - pd.to_timedelta(dates.dt.weekday, unit="D")


def build_weekly_series(
    df_raw: pd.DataFrame,
    date_col: str,
    unique_id: str,
) -> pd.DataFrame:
    """
    Construye una serie semanal de conteos en formato Nixtla.

    Si el archivo ya viene en formato Nixtla, solo normaliza tipos y orden.
    Si viene como datos crudos con una columna de fecha, agrupa por semana.
    """
    required_nixtla_cols = {"unique_id", "ds", "y"}

    if required_nixtla_cols.issubset(df_raw.columns):
        df_weekly = df_raw[["unique_id", "ds", "y"]].copy()
        df_weekly["ds"] = pd.to_datetime(df_weekly["ds"], errors="coerce")
        df_weekly["y"] = pd.to_numeric(df_weekly["y"], errors="coerce")
        df_weekly = df_weekly.dropna(subset=["ds", "y"])
        return df_weekly.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    if date_col not in df_raw.columns:
        raise ValueError(
            f"No existe la columna '{date_col}'. Columnas disponibles: {list(df_raw.columns)}"
        )

    df = df_raw.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col])

    df["ds"] = monday_of_week(df[date_col])

    df_weekly = (
        df.groupby("ds", as_index=False)
          .size()
          .rename(columns={"size": "y"})
          .assign(unique_id=unique_id)
          [["unique_id", "ds", "y"]]
          .sort_values("ds")
          .reset_index(drop=True)
    )

    return df_weekly


def summarize_series(df: pd.DataFrame) -> None:
    """
    Imprime un resumen básico de la serie en formato Nixtla.
    """
    print("=" * 70)
    print("RESUMEN DE LA SERIE TEMPORAL")
    print("=" * 70)
    print(f"Serie: {df['unique_id'].iloc[0]}")
    print(f"Observaciones: {len(df)}")
    print(f"Fecha inicial: {df['ds'].min().date()}")
    print(f"Fecha final:   {df['ds'].max().date()}")
    print(f"Casos mínimos: {df['y'].min():.0f}")
    print(f"Casos máximos: {df['y'].max():.0f}")
    print(f"Media semanal: {df['y'].mean():.2f}")
    print(f"Mediana:       {df['y'].median():.2f}")
    print("=" * 70)

In [37]:
# ============================================================
# 3.2 Calidad de datos, calendario completo e imputación
# ============================================================

def complete_weekly_calendar(df: pd.DataFrame, freq: str = "W-MON") -> pd.DataFrame:
    """
    Crea un calendario semanal completo entre la primera y última fecha observada.
    Los huecos quedan como NaN en la columna y.
    """
    unique_id = df["unique_id"].iloc[0]

    expected_dates = pd.date_range(
        start=df["ds"].min(),
        end=df["ds"].max(),
        freq=freq,
    )

    calendar = pd.DataFrame({
        "unique_id": unique_id,
        "ds": expected_dates,
    })

    completed = (
        calendar.merge(df, on=["unique_id", "ds"], how="left")
                .sort_values("ds")
                .reset_index(drop=True)
    )

    return completed


def diagnose_missing_values(df: pd.DataFrame, freq: str = "W-MON") -> pd.DataFrame:
    """
    Diagnostica continuidad temporal y valores faltantes.
    """
    expected_dates = pd.date_range(df["ds"].min(), df["ds"].max(), freq=freq)
    present_dates = pd.DatetimeIndex(df["ds"])
    missing_dates = expected_dates.difference(present_dates)

    summary = pd.DataFrame({
        "indicador": [
            "observaciones_esperadas",
            "observaciones_presentes",
            "fechas_faltantes",
            "nan_en_y",
            "duplicados_fecha",
        ],
        "valor": [
            len(expected_dates),
            len(df),
            len(missing_dates),
            int(df["y"].isna().sum()),
            int(df.duplicated(subset=["unique_id", "ds"]).sum()),
        ],
    })

    display(summary)

    if len(missing_dates) > 0:
        print("Fechas faltantes detectadas:")
        display(pd.DataFrame({"ds_faltante": missing_dates}))

    return summary


def temporal_train_test_split(
    df: pd.DataFrame,
    train_fraction: float = 0.70,
    cutoff_date: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Timestamp]:
    """
    Divide la serie respetando el orden temporal.
    Puede usar una fecha explícita o una fracción de entrenamiento.
    """
    df = df.sort_values("ds").reset_index(drop=True)

    if cutoff_date is None:
        cutoff_index = int(len(df) * train_fraction)
        cutoff = df.loc[cutoff_index, "ds"]
    else:
        cutoff = pd.Timestamp(cutoff_date)

    train = df[df["ds"] < cutoff].copy()
    test = df[df["ds"] >= cutoff].copy()

    return train, test, cutoff


def seasonal_imputation_from_train(
    train: pd.DataFrame,
    test: pd.DataFrame,
    value_col: str = "y",
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """
    Imputa valores faltantes usando medias estacionales calculadas SOLO con train.

    Esto evita usar información del conjunto de prueba durante el preprocesamiento.
    """
    train_imp = train.copy()
    test_imp = test.copy()

    train_imp["week_of_year"] = train_imp["ds"].dt.isocalendar().week.astype(int)
    test_imp["week_of_year"] = test_imp["ds"].dt.isocalendar().week.astype(int)

    seasonal_means = (
        train_imp.groupby("week_of_year")[value_col]
                 .mean()
                 .to_dict()
    )

    global_train_mean = train_imp[value_col].mean()

    def fill_value(row):
        if pd.notna(row[value_col]):
            return row[value_col]

        week = int(row["week_of_year"])
        return seasonal_means.get(week, global_train_mean)

    train_imp[value_col] = train_imp.apply(fill_value, axis=1)
    test_imp[value_col] = test_imp.apply(fill_value, axis=1)

    train_imp = train_imp.drop(columns=["week_of_year"])
    test_imp = test_imp.drop(columns=["week_of_year"])

    metadata = {
        "seasonal_means": seasonal_means,
        "global_train_mean": global_train_mean,
        "train_missing_before": int(train[value_col].isna().sum()),
        "test_missing_before": int(test[value_col].isna().sum()),
        "train_missing_after": int(train_imp[value_col].isna().sum()),
        "test_missing_after": int(test_imp[value_col].isna().sum()),
    }

    return train_imp, test_imp, metadata


def combine_train_test(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    """
    Une train y test ya procesados para visualización.
    """
    return (
        pd.concat([train, test], ignore_index=True)
          .sort_values("ds")
          .reset_index(drop=True)
    )

In [38]:
# ============================================================
# 3.3 Visualización y diagnóstico descriptivo
# ============================================================

def plot_time_series(
    df: pd.DataFrame,
    title: str,
    y_col: str = "y",
    line_name: str = "Serie",
    height: int = 430,
) -> go.Figure:
    """
    Grafica una serie temporal simple.
    """
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["ds"],
            y=df[y_col],
            mode="lines+markers",
            name=line_name,
            line=dict(width=1.8),
            marker=dict(size=4),
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="Fecha",
        yaxis_title="Casos",
        height=height,
        template="plotly_white",
    )

    return fig


def plot_train_test_split(
    train: pd.DataFrame,
    test: pd.DataFrame,
    cutoff: pd.Timestamp,
    title: str = "División temporal train/test",
) -> go.Figure:
    """
    Grafica la división temporal del conjunto de entrenamiento y prueba.
    """
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=train["ds"],
            y=train["y"],
            mode="lines+markers",
            name="Train",
            marker=dict(size=3),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=test["ds"],
            y=test["y"],
            mode="lines+markers",
            name="Test",
            marker=dict(size=3),
        )
    )

    fig.add_shape(
        type="line",
        xref="x",
        yref="paper",
        x0=cutoff,
        x1=cutoff,
        y0=0,
        y1=1,
        line=dict(dash="dash", width=1.5),
    )

    fig.add_annotation(
        x=cutoff,
        y=1.03,
        yref="paper",
        text="Corte",
        showarrow=False,
        xanchor="left",
    )

    fig.update_layout(
        title=title,
        xaxis_title="Fecha",
        yaxis_title="Casos",
        height=430,
        template="plotly_white",
    )

    return fig


def detect_iqr_outliers(df: pd.DataFrame, value_col: str = "y") -> tuple[pd.DataFrame, dict]:
    """
    Detecta outliers mediante IQR global.
    Esta detección es diagnóstica: no elimina datos automáticamente.
    """
    q1 = df[value_col].quantile(0.25)
    q3 = df[value_col].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outliers = df[(df[value_col] < lower_limit) | (df[value_col] > upper_limit)].copy()

    metadata = {
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_limit": lower_limit,
        "upper_limit": upper_limit,
        "n_outliers": len(outliers),
    }

    return outliers, metadata


def detect_monthly_iqr_outliers(df: pd.DataFrame, value_col: str = "y") -> pd.DataFrame:
    """
    Detecta outliers con IQR dentro de cada mes calendario.
    Es más razonable que el IQR global cuando existe estacionalidad.
    """
    df_check = df.copy()
    df_check["month"] = df_check["ds"].dt.month
    df_check["outlier_monthly_iqr"] = False

    for month in range(1, 13):
        mask = df_check["month"] == month
        values = df_check.loc[mask, value_col]

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr

        is_outlier = (values < lower_limit) | (values > upper_limit)
        df_check.loc[mask & is_outlier, "outlier_monthly_iqr"] = True

    return df_check[df_check["outlier_monthly_iqr"]].copy()


def plot_iqr_outliers(
    df: pd.DataFrame,
    outliers: pd.DataFrame,
    metadata: dict,
    title: str = "Detección diagnóstica de outliers — IQR global",
) -> go.Figure:
    """
    Grafica la serie y marca los outliers detectados por IQR.
    """
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["ds"],
            y=df["y"],
            mode="lines",
            name="Serie",
            line=dict(width=1.5),
        )
    )

    fig.add_hline(
        y=metadata["upper_limit"],
        line_dash="dash",
        annotation_text="Límite superior IQR",
    )

    fig.add_hline(
        y=metadata["lower_limit"],
        line_dash="dash",
        annotation_text="Límite inferior IQR",
    )

    if len(outliers) > 0:
        fig.add_trace(
            go.Scatter(
                x=outliers["ds"],
                y=outliers["y"],
                mode="markers",
                name="Outlier IQR",
                marker=dict(size=10, symbol="circle-open", line_width=2),
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="Fecha",
        yaxis_title="Casos",
        height=410,
        template="plotly_white",
    )

    return fig


def plot_rolling_statistics(
    df: pd.DataFrame,
    window: int = 8,
    title: str = "Media y desviación estándar móviles",
) -> go.Figure:
    """
    Grafica media y desviación estándar móviles para revisar cambios de régimen.
    """
    rolling_mean = df["y"].rolling(window=window, center=True).mean()
    rolling_std = df["y"].rolling(window=window, center=True).std()

    fig = make_subplots(
        rows=2,
        cols=1,
        subplot_titles=[
            f"Serie y media móvil, ventana={window}",
            f"Desviación estándar móvil, ventana={window}",
        ],
        vertical_spacing=0.15,
    )

    fig.add_trace(
        go.Scatter(x=df["ds"], y=df["y"], mode="lines", name="Serie"),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(x=df["ds"], y=rolling_mean, mode="lines", name="Media móvil"),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(x=df["ds"], y=rolling_std, mode="lines", name="Desv. estándar móvil"),
        row=2,
        col=1,
    )

    fig.update_layout(
        title=title,
        height=570,
        template="plotly_white",
    )

    return fig

In [39]:
# ============================================================
# 3.4 Autocorrelación, estacionariedad y transformaciones
# ============================================================

def clean_numeric_series(series: pd.Series) -> np.ndarray:
    """
    Convierte una serie a arreglo numérico sin NaN.
    """
    values = pd.to_numeric(series, errors="coerce").dropna().astype(float).values
    return values


def plot_acf_plotly(
    series: pd.Series,
    title: str,
    n_lags: int = 52,
) -> go.Figure:
    """
    Grafica la ACF con intervalo de confianza aproximado al 95%.
    """
    values = clean_numeric_series(series)
    acf_values = acf(values, nlags=n_lags, fft=True)

    ci = 1.96 / np.sqrt(len(values))
    lags = np.arange(len(acf_values))

    fig = go.Figure()

    for lag, value in zip(lags, acf_values):
        fig.add_trace(
            go.Scatter(
                x=[lag, lag],
                y=[0, value],
                mode="lines",
                line=dict(width=2.4),
                showlegend=False,
            )
        )

    fig.add_trace(
        go.Scatter(
            x=lags,
            y=acf_values,
            mode="markers",
            showlegend=False,
            marker=dict(size=6),
        )
    )

    fig.add_hline(y=ci, line_dash="dash", annotation_text=f"IC 95% ≈ ±{ci:.3f}")
    fig.add_hline(y=-ci, line_dash="dash")
    fig.add_hline(y=0, line_width=0.8)

    fig.update_layout(
        title=title,
        xaxis_title="Lag",
        yaxis_title="Autocorrelación",
        height=390,
        template="plotly_white",
        yaxis=dict(range=[-1.05, 1.05]),
    )

    return fig


def stationarity_tests(series: pd.Series, name: str) -> dict:
    """
    Aplica ADF y KPSS y devuelve una conclusión conjunta.
    """
    values = clean_numeric_series(series)

    if len(values) < 12:
        raise ValueError(f"La serie '{name}' tiene muy pocos datos para pruebas de estacionariedad.")

    adf_stat, adf_p, _, _, _, _ = adfuller(values, autolag="AIC")
    kpss_stat, kpss_p, _, _ = kpss(values, regression="c", nlags="auto")

    adf_stationary = adf_p < 0.05
    kpss_stationary = kpss_p >= 0.05

    if adf_stationary and kpss_stationary:
        conclusion = "Estacionaria"
    elif (not adf_stationary) and (not kpss_stationary):
        conclusion = "No estacionaria"
    else:
        conclusion = "Resultado mixto o incierto"

    result = {
        "serie": name,
        "adf_stat": adf_stat,
        "adf_pvalue": adf_p,
        "kpss_stat": kpss_stat,
        "kpss_pvalue": kpss_p,
        "conclusion": conclusion,
    }

    return result


def run_stationarity_suite(series_dict: dict[str, pd.Series]) -> pd.DataFrame:
    """
    Ejecuta ADF y KPSS sobre varias versiones de una serie.
    """
    results = []

    for name, series in series_dict.items():
        result = stationarity_tests(series, name)
        results.append(result)

    return pd.DataFrame(results)


def ljung_box_table(series: pd.Series, lags: list[int]) -> pd.DataFrame:
    """
    Ejecuta Ljung-Box para varios lags.
    """
    values = clean_numeric_series(series)
    rows = []

    for lag in lags:
        result = acorr_ljungbox(values, lags=[lag], return_df=True)
        p_value = result["lb_pvalue"].iloc[0]
        q_stat = result["lb_stat"].iloc[0]

        rows.append({
            "lag": lag,
            "lb_stat": q_stat,
            "p_value": p_value,
            "conclusion": (
                "Hay autocorrelación" if p_value < 0.05 else "No hay evidencia de autocorrelación"
            ),
        })

    return pd.DataFrame(rows)


def add_transformations(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrega transformaciones comunes para diagnóstico:
    - log1p(y)
    - diff(y)
    - diff(log1p(y))
    """
    transformed = df.copy()

    transformed["y_log1p"] = np.log1p(transformed["y"])
    transformed["y_diff"] = transformed["y"].diff()
    transformed["y_log1p_diff"] = transformed["y_log1p"].diff()

    return transformed


def plot_transformations(df: pd.DataFrame) -> go.Figure:
    """
    Grafica serie original y transformaciones principales.
    """
    fig = make_subplots(
        rows=4,
        cols=1,
        subplot_titles=[
            "Serie original: y",
            "Transformación logarítmica: log1p(y)",
            "Primera diferencia: diff(y)",
            "Diferencia logarítmica: diff(log1p(y))",
        ],
        vertical_spacing=0.08,
    )

    columns = ["y", "y_log1p", "y_diff", "y_log1p_diff"]

    for row, column in enumerate(columns, start=1):
        fig.add_trace(
            go.Scatter(
                x=df["ds"],
                y=df[column],
                mode="lines",
                name=column,
            ),
            row=row,
            col=1,
        )

    fig.update_layout(
        title="Comparación visual de transformaciones",
        height=820,
        template="plotly_white",
        showlegend=False,
    )

    return fig

In [ ]:
# ============================================================
# 3.5 Modelos baseline, métricas, residuos y validación cruzada
# ============================================================

def make_baseline_models(season_length: int = 52) -> list:
    """
    Define los modelos baseline usados en el taller.
    """
    return [
        Naive(),
        SeasonalNaive(season_length=season_length),
        WindowAverage(window_size=3),
        RandomWalkWithDrift(),
    ]


def fit_predict_baselines(
    train: pd.DataFrame,
    h: int,
    freq: str = "W-MON",
    season_length: int = 52,
) -> tuple[StatsForecast, pd.DataFrame]:
    """
    Entrena modelos baseline y genera pronósticos.
    """
    sf = StatsForecast(
        models=make_baseline_models(season_length=season_length),
        freq=freq,
    )

    sf.fit(train)
    forecasts = sf.predict(h=h)

    return sf, forecasts


def merge_forecasts_with_test(test: pd.DataFrame, forecasts: pd.DataFrame) -> pd.DataFrame:
    """
    Une los pronósticos con los valores reales del periodo de prueba.
    """
    return test.merge(forecasts, on=["unique_id", "ds"], how="inner")


def get_model_columns(df_forecasts: pd.DataFrame) -> list[str]:
    """
    Identifica columnas de modelos en un DataFrame de pronósticos.
    """
    excluded = {"unique_id", "ds", "cutoff", "y"}
    return [column for column in df_forecasts.columns if column not in excluded]


def evaluate_baselines(
    test_forecasts: pd.DataFrame,
    train: pd.DataFrame,
    models: list[str],
) -> pd.DataFrame:
    """
    Calcula MAE, RMSE, sMAPE y MASE.

    MASE usa como denominador el MAE del naive de un paso sobre train.
    """
    evaluation = evaluate(
        test_forecasts,
        metrics=[mae, rmse, smape],
        models=models,
        target_col="y",
    )

    naive_train_mae = np.abs(np.diff(train["y"].values)).mean()

    mase_row = evaluation[evaluation["metric"] == "mae"].copy()
    mase_row["metric"] = "mase"

    for model in models:
        mase_row[model] = mase_row[model] / naive_train_mae

    evaluation = pd.concat([evaluation, mase_row], ignore_index=True)

    return evaluation


def best_model_by_metric(evaluation: pd.DataFrame, metric: str, models: list[str]) -> tuple[str, float]:
    """
    Encuentra el mejor modelo para una métrica específica.
    """
    row = evaluation[evaluation["metric"] == metric]

    if row.empty:
        raise ValueError(f"No existe la métrica '{metric}' en la tabla de evaluación.")

    values = row[models].iloc[0]
    best_model = values.astype(float).idxmin()
    best_value = float(values[best_model])

    return best_model, best_value


def print_best_models(evaluation: pd.DataFrame, models: list[str]) -> None:
    """
    Imprime el mejor modelo por métrica.
    """
    print("Mejor modelo por métrica")
    print("=" * 50)

    for metric in evaluation["metric"].unique():
        best_model, best_value = best_model_by_metric(evaluation, metric, models)
        print(f"{metric.upper():>6}: {best_model:<18} {best_value:.4f}")


def plot_forecasts(
    df_full: pd.DataFrame,
    train: pd.DataFrame,
    test_forecasts: pd.DataFrame,
    cutoff: pd.Timestamp,
    models: list[str],
) -> go.Figure:
    """
    Grafica serie histórica, valores reales de test y pronósticos baseline.
    """
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df_full["ds"],
            y=df_full["y"],
            mode="lines",
            name="Histórico",
            line=dict(width=1),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=test_forecasts["ds"],
            y=test_forecasts["y"],
            mode="lines+markers",
            name="Real test",
            marker=dict(size=3),
            line=dict(width=2),
        )
    )

    for model in models:
        fig.add_trace(
            go.Scatter(
                x=test_forecasts["ds"],
                y=test_forecasts[model],
                mode="lines+markers",
                name=model,
                marker=dict(size=3),
                line=dict(width=1.8, dash="dot"),
            )
        )

    fig.add_shape(
        type="line",
        xref="x",
        yref="paper",
        x0=cutoff,
        x1=cutoff,
        y0=0,
        y1=1,
        line=dict(dash="dash", width=1.3),
    )

    fig.update_layout(
        title="Comparación de modelos baseline sobre el conjunto de prueba",
        xaxis_title="Fecha",
        yaxis_title="Casos",
        height=500,
        template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )

    return fig


def plot_metric_comparison(evaluation: pd.DataFrame, models: list[str]) -> go.Figure:
    """
    Grafica MAE, RMSE y sMAPE para comparar modelos.
    """
    metrics_to_plot = ["mae", "rmse", "smape"]
    fig = make_subplots(
        rows=1,
        cols=len(metrics_to_plot),
        subplot_titles=[metric.upper() for metric in metrics_to_plot],
    )

    for col_index, metric in enumerate(metrics_to_plot, start=1):
        row = evaluation[evaluation["metric"] == metric]
        values = [float(row[model].iloc[0]) for model in models]

        fig.add_trace(
            go.Bar(
                x=models,
                y=values,
                text=[f"{value:.3f}" for value in values],
                textposition="outside",
                showlegend=False,
            ),
            row=1,
            col=col_index,
        )

    fig.update_layout(
        title="Comparación de métricas de modelos baseline",
        height=420,
        template="plotly_white",
    )

    return fig


def residual_diagnostics(
    test_forecasts: pd.DataFrame,
    model_name: str,
    lags: list[int] = [1, 6, 12, 24],
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Calcula residuos y aplica Ljung-Box a los residuos.
    """
    residuals = test_forecasts["y"] - test_forecasts[model_name]
    ljung_residuals = ljung_box_table(residuals, lags=lags)

    return residuals, ljung_residuals


def plot_residuals(
    test_forecasts: pd.DataFrame,
    residuals: pd.Series,
    model_name: str,
    n_lags: int = 52,
) -> go.Figure:
    """
    Grafica residuos en el tiempo y su ACF.
    """
    residual_values = clean_numeric_series(residuals)
    acf_residuals = acf(residual_values, nlags=min(n_lags, len(residual_values) - 1), fft=True)
    ci = 1.96 / np.sqrt(len(residual_values))

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            f"Residuos en el tiempo — {model_name}",
            f"ACF de residuos — {model_name}",
        ],
    )

    fig.add_trace(
        go.Scatter(
            x=test_forecasts["ds"],
            y=residuals,
            mode="lines+markers",
            name="Residuos",
            marker=dict(size=4),
        ),
        row=1,
        col=1,
    )

    fig.add_hline(y=0, line_dash="dash", row=1, col=1)

    for lag, value in enumerate(acf_residuals):
        fig.add_trace(
            go.Scatter(
                x=[lag, lag],
                y=[0, value],
                mode="lines",
                line=dict(width=2.5),
                showlegend=False,
            ),
            row=1,
            col=2,
        )

    fig.add_hline(y=ci, line_dash="dash", row=1, col=2)
    fig.add_hline(y=-ci, line_dash="dash", row=1, col=2)

    fig.update_layout(
        title=f"Diagnóstico de residuos del modelo {model_name}",
        height=420,
        template="plotly_white",
        showlegend=False,
    )

    return fig


def run_time_series_cross_validation(
    df: pd.DataFrame,
    freq: str = "W-MON",
    season_length: int = 52,
    h: int = 52,
    n_windows: int = 3,
    step_size: int = 52,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Ejecuta validación cruzada temporal con StatsForecast.
    """
    sf_cv = StatsForecast(
        models=make_baseline_models(season_length=season_length),
        freq=freq,
    )

    cv_results = sf_cv.cross_validation(
        df=df,
        h=h,
        n_windows=n_windows,
        step_size=step_size,
    )

    cv_models = get_model_columns(cv_results)

    cv_evaluation = evaluate(
        cv_results,
        metrics=[mae, rmse, smape],
        models=cv_models,
        target_col="y",
    )

    return cv_results, cv_evaluation, cv_models

---

## 4. Carga de datos y construcción de la serie semanal

En esta sección se cargan los datos crudos de dengue y se transforman en una serie temporal semanal.

Para evitar problemas con semanas ISO como `2009-W53`, la fecha semanal se construye llevando cada fecha observada al lunes de su semana correspondiente. Así el campo `ds` queda como una fecha real y no como un string de año-semana.

In [41]:
df_raw = load_csv_data(DATA_PATH)

display(df_raw.head())
display(pd.DataFrame({"columnas": df_raw.columns}))

⚠️ DATA_PATH no existe. Se usará el archivo encontrado: dengue_semanal_nixtla.csv
Archivo cargado: dengue_semanal_nixtla.csv
Dimensiones: 741 filas × 3 columnas


,unique_id,ds,y
0,dengue_cali,2009-12-28,12
1,dengue_cali,2010-01-04,142
2,dengue_cali,2010-01-11,210
3,dengue_cali,2010-01-18,253
4,dengue_cali,2010-01-25,345


,columnas
0,unique_id
1,ds
2,y


In [42]:
df_weekly = build_weekly_series(
    df_raw=df_raw,
    date_col=DATE_COLUMN,
    unique_id=SERIES_ID,
)

summarize_series(df_weekly)
display(df_weekly.head(10))
display(df_weekly.tail(10))

RESUMEN DE LA SERIE TEMPORAL
Serie: dengue_cali
Observaciones: 741
Fecha inicial: 2009-12-28
Fecha final:   2024-03-25
Casos mínimos: 1
Casos máximos: 573
Media semanal: 83.58
Mediana:       51.00


,unique_id,ds,y
0,dengue_cali,2009-12-28,12
1,dengue_cali,2010-01-04,142
2,dengue_cali,2010-01-11,210
3,dengue_cali,2010-01-18,253
4,dengue_cali,2010-01-25,345
5,dengue_cali,2010-02-01,375
6,dengue_cali,2010-02-08,504
7,dengue_cali,2010-02-15,573
8,dengue_cali,2010-02-22,543
9,dengue_cali,2010-03-01,475


,unique_id,ds,y
731,dengue_cali,2024-01-01,213
732,dengue_cali,2024-01-08,42
733,dengue_cali,2024-01-15,13
734,dengue_cali,2024-01-22,3
735,dengue_cali,2024-01-29,6
736,dengue_cali,2024-02-05,4
737,dengue_cali,2024-02-12,7
738,dengue_cali,2024-02-19,6
739,dengue_cali,2024-03-04,5
740,dengue_cali,2024-03-25,1


In [43]:
output_path = "/content/dengue_semanal_nixtla_reorganizado.csv"

try:
    df_weekly.to_csv(output_path, index=False)
    print(f"Archivo semanal guardado en: {output_path}")
except Exception as error:
    print(f"No se pudo guardar en /content. Motivo: {error}")

No se pudo guardar en /content. Motivo: Cannot save file into a non-existent directory: '/content'


---

## 5. Exploración visual inicial de la serie

Antes de entrenar modelos, conviene observar la serie en escala original.  
En enfermedades como dengue, los picos pueden representar brotes reales; por eso esta visualización ayuda a distinguir posibles anomalías estadísticas de eventos epidemiológicos relevantes.

In [44]:
fig = plot_time_series(
    df_weekly,
    title="Casos semanales de dengue — Serie original",
    line_name="Casos semanales",
)

fig.show()

---

## 6. Diagnóstico de calidad de datos

Esta sección revisa:

- continuidad semanal;
- fechas faltantes;
- valores nulos en `y`;
- duplicados por fecha.

Después se crea un calendario semanal completo.  
Los valores faltantes todavía no se imputan aquí, porque primero se realizará el split temporal. Esto evita que el conjunto de prueba influya en las reglas de imputación.

In [45]:
print("Diagnóstico sobre la serie semanal observada:")
diagnose_missing_values(df_weekly, freq=FREQ)

df_complete = complete_weekly_calendar(df_weekly, freq=FREQ)

print("\nDiagnóstico después de crear calendario semanal completo:")
diagnose_missing_values(df_complete, freq=FREQ)

display(df_complete.head())
display(df_complete.tail())

Diagnóstico sobre la serie semanal observada:


,indicador,valor
0,observaciones_esperadas,744
1,observaciones_presentes,741
2,fechas_faltantes,3
3,nan_en_y,0
4,duplicados_fecha,0


Fechas faltantes detectadas:


,ds_faltante
0,2024-02-26
1,2024-03-11
2,2024-03-18



Diagnóstico después de crear calendario semanal completo:


,indicador,valor
0,observaciones_esperadas,744
1,observaciones_presentes,744
2,fechas_faltantes,0
3,nan_en_y,3
4,duplicados_fecha,0


,unique_id,ds,y
0,dengue_cali,2009-12-28,12.0
1,dengue_cali,2010-01-04,142.0
2,dengue_cali,2010-01-11,210.0
3,dengue_cali,2010-01-18,253.0
4,dengue_cali,2010-01-25,345.0


,unique_id,ds,y
739,dengue_cali,2024-02-26,NaN
740,dengue_cali,2024-03-04,5.0
741,dengue_cali,2024-03-11,NaN
742,dengue_cali,2024-03-18,NaN
743,dengue_cali,2024-03-25,1.0


---

## 7. Diagnóstico de outliers y anomalías

La detección de outliers se usa como diagnóstico, no como eliminación automática.

En series epidemiológicas, un valor extremo puede ser un error de registro, pero también puede representar un brote real. Por eso se reportan outliers globales y outliers ajustados por mes, pero la serie se conserva completa para los modelos baseline.

In [46]:
outliers_iqr, outlier_metadata = detect_iqr_outliers(df_complete.dropna(subset=["y"]))

print("IQR global")
print("=" * 50)
for key, value in outlier_metadata.items():
    print(f"{key}: {value:.3f}" if isinstance(value, float) else f"{key}: {value}")

display(outliers_iqr[["unique_id", "ds", "y"]])

fig = plot_iqr_outliers(
    df_complete.dropna(subset=["y"]),
    outliers_iqr,
    outlier_metadata,
)

fig.show()

IQR global
q1: 17.000
q3: 107.000
iqr: 90.000
lower_limit: -118.000
upper_limit: 242.000
n_outliers: 57


,unique_id,ds,y
3,dengue_cali,2010-01-18,253.0
4,dengue_cali,2010-01-25,345.0
5,dengue_cali,2010-02-01,375.0
6,dengue_cali,2010-02-08,504.0
7,dengue_cali,2010-02-15,573.0
8,dengue_cali,2010-02-22,543.0
9,dengue_cali,2010-03-01,475.0
10,dengue_cali,2010-03-08,451.0
11,dengue_cali,2010-03-15,439.0
12,dengue_cali,2010-03-22,384.0


In [47]:
monthly_outliers = detect_monthly_iqr_outliers(df_complete.dropna(subset=["y"]))

print(f"Outliers detectados con IQR por mes: {len(monthly_outliers)}")
display(monthly_outliers[["unique_id", "ds", "y", "month"]].head(20))

Outliers detectados con IQR por mes: 56


,unique_id,ds,y,month
3,dengue_cali,2010-01-18,253.0,1
4,dengue_cali,2010-01-25,345.0,1
5,dengue_cali,2010-02-01,375.0,2
6,dengue_cali,2010-02-08,504.0,2
7,dengue_cali,2010-02-15,573.0,2
8,dengue_cali,2010-02-22,543.0,2
9,dengue_cali,2010-03-01,475.0,3
10,dengue_cali,2010-03-08,451.0,3
11,dengue_cali,2010-03-15,439.0,3
12,dengue_cali,2010-03-22,384.0,3


In [48]:
fig = plot_rolling_statistics(
    df_complete.dropna(subset=["y"]),
    window=8,
    title="Revisión visual de posibles cambios de régimen",
)

fig.show()

---

## 8. División temporal train/test

La división se realiza respetando el orden temporal de la serie.  
No se mezclan observaciones aleatoriamente porque en series temporales el pasado debe predecir el futuro.

La imputación se hace después de este corte, usando únicamente las medias estacionales calculadas con el conjunto de entrenamiento.

In [49]:
train_raw, test_raw, cutoff = temporal_train_test_split(
    df=df_complete,
    train_fraction=TRAIN_FRACTION,
    cutoff_date=CUTOFF_DATE,
)

print(f"TRAIN: {len(train_raw)} observaciones")
print(f"  Desde: {train_raw['ds'].min().date()} hasta {train_raw['ds'].max().date()}")

print(f"\nTEST: {len(test_raw)} observaciones")
print(f"  Desde: {test_raw['ds'].min().date()} hasta {test_raw['ds'].max().date()}")

print(f"\nFecha de corte: {cutoff.date()}")

fig = plot_train_test_split(
    train_raw,
    test_raw,
    cutoff,
    title=f"Train/test split temporal — {TRAIN_FRACTION:.0%}/{1-TRAIN_FRACTION:.0%}",
)

fig.show()

TRAIN: 523 observaciones
  Desde: 2009-12-28 hasta 2019-12-30

TEST: 221 observaciones
  Desde: 2020-01-06 hasta 2024-03-25

Fecha de corte: 2020-01-01


---

## 9. Imputación estacional sin fuga de información

Si existen semanas faltantes, se imputan usando la media histórica de la misma semana del año calculada solamente con `train`.

Esto mantiene el principio de evaluación temporal: el conjunto de prueba no debe ayudar a construir reglas de preprocesamiento.

In [50]:
train, test, imputation_metadata = seasonal_imputation_from_train(
    train=train_raw,
    test=test_raw,
    value_col="y",
)

print("Resumen de imputación")
print("=" * 50)
for key, value in imputation_metadata.items():
    if key != "seasonal_means":
        print(f"{key}: {value}")

df_model = combine_train_test(train, test)

print("\nVerificación final de faltantes:")
diagnose_missing_values(df_model, freq=FREQ)

fig = plot_train_test_split(
    train,
    test,
    cutoff,
    title="Train/test después de imputación estacional",
)

fig.show()

Resumen de imputación
global_train_mean: 73.52772466539197
train_missing_before: 0
test_missing_before: 3
train_missing_after: 0
test_missing_after: 0

Verificación final de faltantes:


,indicador,valor
0,observaciones_esperadas,744
1,observaciones_presentes,744
2,fechas_faltantes,0
3,nan_en_y,0
4,duplicados_fecha,0


---

## 10. Modelos baseline sobre la serie original

Los modelos baseline se entrenan sobre la serie en escala original.  
Esto permite obtener una referencia simple e interpretable antes de aplicar modelos más sofisticados.

Modelos incluidos:

- `Naive`: predice el último valor observado.
- `SeasonalNaive`: predice el valor de la misma semana del año anterior.
- `WindowAverage`: usa el promedio de las últimas observaciones.
- `RandomWalkWithDrift`: extiende una caminata aleatoria con tendencia.

In [51]:
h = len(test)

sf, forecasts = fit_predict_baselines(
    train=train,
    h=h,
    freq=FREQ,
    season_length=SEASON_LENGTH,
)

test_forecasts = merge_forecasts_with_test(test, forecasts)
model_columns = get_model_columns(test_forecasts)

print("Modelos entrenados:")
print(model_columns)
print(f"Horizonte de pronóstico: {h} semanas")

display(test_forecasts.head())

Modelos entrenados:
['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']
Horizonte de pronóstico: 221 semanas


,unique_id,ds,y,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,2020-01-06,180.0,162.0,12.0,141.666667,162.287356
1,dengue_cali,2020-01-13,264.0,162.0,11.0,141.666667,162.574713
2,dengue_cali,2020-01-20,192.0,162.0,3.0,141.666667,162.862069
3,dengue_cali,2020-01-27,438.0,162.0,5.0,141.666667,163.149425
4,dengue_cali,2020-02-03,348.0,162.0,9.0,141.666667,163.436782


In [52]:
fig = plot_forecasts(
    df_full=df_model,
    train=train,
    test_forecasts=test_forecasts,
    cutoff=cutoff,
    models=model_columns,
)

fig.show()

---

## 11. Evaluación de modelos baseline

Se calculan métricas de error en el conjunto de prueba:

- `MAE`: error absoluto medio.
- `RMSE`: penaliza más los errores grandes.
- `sMAPE`: error porcentual simétrico.
- `MASE`: error escalado respecto a un naive de entrenamiento.

La evaluación se hace en escala original, es decir, en número de casos.

In [53]:
evaluation = evaluate_baselines(
    test_forecasts=test_forecasts,
    train=train,
    models=model_columns,
)

display(evaluation)

print_best_models(evaluation, model_columns)

,unique_id,metric,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,mae,111.987330,87.342986,99.925490,130.194695
1,dengue_cali,rmse,123.140951,131.788126,115.617042,140.111914
2,dengue_cali,smape,0.455584,0.524727,0.428548,0.482878
3,dengue_cali,mase,8.894916,6.937468,7.936869,10.341088


Mejor modelo por métrica
   MAE: SeasonalNaive      87.3430
  RMSE: WindowAverage      115.6170
 SMAPE: WindowAverage      0.4285
  MASE: SeasonalNaive      6.9375


In [54]:
fig = plot_metric_comparison(evaluation, model_columns)
fig.show()

---

## 12. Diagnóstico de residuos del mejor baseline

Después de elegir el mejor modelo según una métrica principal, se analizan sus residuos.

Aquí se usa `MAE` como criterio principal por ser una métrica directa e interpretable en número de casos.  
Si los residuos conservan autocorrelación, significa que el baseline todavía deja patrones sin modelar.

In [55]:
PRIMARY_METRIC = "mae"

best_model, best_value = best_model_by_metric(
    evaluation=evaluation,
    metric=PRIMARY_METRIC,
    models=model_columns,
)

print(f"Mejor modelo según {PRIMARY_METRIC.upper()}: {best_model}")
print(f"Valor de {PRIMARY_METRIC.upper()}: {best_value:.4f}")

residuals, residual_ljung = residual_diagnostics(
    test_forecasts=test_forecasts,
    model_name=best_model,
    lags=[1, 6, 12, 24],
)

print("\nResumen de residuos:")
print(f"Media: {residuals.mean():.3f}")
print(f"Desviación estándar: {residuals.std():.3f}")

display(residual_ljung)

fig = plot_residuals(
    test_forecasts=test_forecasts,
    residuals=residuals,
    model_name=best_model,
)

fig.show()

Mejor modelo según MAE: SeasonalNaive
Valor de MAE: 87.3430

Resumen de residuos:
Media: 70.773
Desviación estándar: 111.425


,lag,lb_stat,p_value,conclusion
0,1,191.351960,1.611509e-43,Hay autocorrelación
1,6,867.601913,3.786097e-184,Hay autocorrelación
2,12,1205.463370,1.153468e-250,Hay autocorrelación
3,24,1331.535386,2.105636e-266,Hay autocorrelación


---

## 13. Autocorrelación de la serie

La autocorrelación permite revisar cuánto se relaciona la serie consigo misma en rezagos anteriores.

En esta etapa se analiza principalmente el conjunto de entrenamiento, porque es el tramo disponible para decidir el tipo de modelamiento sin mirar el futuro.

In [56]:
fig = plot_acf_plotly(
    train["y"],
    title="ACF — Serie de entrenamiento en escala original",
    n_lags=52,
)

fig.show()

In [57]:
ljung_original = ljung_box_table(
    train["y"],
    lags=[1, 6, 12, 24, 52],
)

display(ljung_original)

,lag,lb_stat,p_value,conclusion
0,1,495.337032,9.829862e-110,Hay autocorrelación
1,6,2483.720322,0.000000e+00,Hay autocorrelación
2,12,3822.491154,0.000000e+00,Hay autocorrelación
3,24,4879.108774,0.000000e+00,Hay autocorrelación
4,52,5051.132800,0.000000e+00,Hay autocorrelación


---

## 14. Pruebas de estacionariedad

Se aplican dos pruebas complementarias:

- **ADF**: su hipótesis nula es que la serie tiene raíz unitaria, es decir, no es estacionaria.
- **KPSS**: su hipótesis nula es que la serie es estacionaria.

La lectura conjunta ayuda a decidir si conviene aplicar transformaciones como logaritmo o diferenciación.

In [58]:
stationarity_original = run_stationarity_suite({
    "train_original": train["y"],
})

display(stationarity_original)

,serie,adf_stat,adf_pvalue,kpss_stat,kpss_pvalue,conclusion
0,train_original,-3.897635,0.002052,0.492219,0.043419,Resultado mixto o incierto


---

## 15. Transformaciones de la serie

En esta sección se prueban transformaciones comunes:

- `log1p(y)`: estabiliza varianza y reduce asimetría.
- `diff(y)`: remueve cambios de nivel o tendencia.
- `diff(log1p(y))`: aproxima cambios relativos entre semanas.

Estas transformaciones se usan como diagnóstico y como preparación para modelos que requieren mayor estacionariedad.  
Los baseline anteriores se conservaron en escala original para que sirvan como referencia interpretable.

In [59]:
train_transformed = add_transformations(train)

fig = plot_transformations(train_transformed)
fig.show()

display(train_transformed.head(10))

,unique_id,ds,y,y_log1p,y_diff,y_log1p_diff
0,dengue_cali,2009-12-28,12.0,2.564949,NaN,NaN
1,dengue_cali,2010-01-04,142.0,4.962845,130.0,2.397895
2,dengue_cali,2010-01-11,210.0,5.351858,68.0,0.389014
3,dengue_cali,2010-01-18,253.0,5.537334,43.0,0.185476
4,dengue_cali,2010-01-25,345.0,5.846439,92.0,0.309105
5,dengue_cali,2010-02-01,375.0,5.929589,30.0,0.083150
6,dengue_cali,2010-02-08,504.0,6.224558,129.0,0.294969
7,dengue_cali,2010-02-15,573.0,6.352629,69.0,0.128071
8,dengue_cali,2010-02-22,543.0,6.298949,-30.0,-0.053680
9,dengue_cali,2010-03-01,475.0,6.165418,-68.0,-0.133531


In [60]:
transformation_stationarity = run_stationarity_suite({
    "original": train_transformed["y"],
    "log1p": train_transformed["y_log1p"],
    "diff": train_transformed["y_diff"],
    "diff_log1p": train_transformed["y_log1p_diff"],
})

display(transformation_stationarity)

,serie,adf_stat,adf_pvalue,kpss_stat,kpss_pvalue,conclusion
0,original,-3.897635,0.002052,0.492219,0.043419,Resultado mixto o incierto
1,log1p,-1.640467,0.462017,0.676394,0.015691,No estacionaria
2,diff,-4.859970,0.000042,0.036081,0.100000,Estacionaria
3,diff_log1p,-24.790540,0.000000,0.090150,0.100000,Estacionaria


In [61]:
fig = plot_acf_plotly(
    train_transformed["y_log1p_diff"],
    title="ACF — Diferencia logarítmica de la serie de entrenamiento",
    n_lags=52,
)

fig.show()

ljung_logdiff = ljung_box_table(
    train_transformed["y_log1p_diff"],
    lags=[1, 6, 12, 24, 52],
)

display(ljung_logdiff)

,lag,lb_stat,p_value,conclusion
0,1,77.219130,1.530044e-18,Hay autocorrelación
1,6,84.300648,4.608061e-16,Hay autocorrelación
2,12,94.437658,6.791564e-15,Hay autocorrelación
3,24,104.171761,5.787860e-12,Hay autocorrelación
4,52,132.959920,5.055379e-09,Hay autocorrelación


---

## 16. Validación cruzada temporal

El split `train/test` ofrece una evaluación clara, pero depende de una única fecha de corte.  
La validación cruzada temporal evalúa los modelos en varias ventanas históricas y permite obtener una comparación más robusta.

En este caso se usa:

- horizonte `h = 52` semanas;
- `3` ventanas de validación;
- desplazamiento anual de `52` semanas.

In [62]:
CV_H = 52
CV_N_WINDOWS = 3
CV_STEP_SIZE = 52

cv_results, cv_evaluation, cv_models = run_time_series_cross_validation(
    df=df_model,
    freq=FREQ,
    season_length=SEASON_LENGTH,
    h=CV_H,
    n_windows=CV_N_WINDOWS,
    step_size=CV_STEP_SIZE,
)

print(f"Resultados CV: {cv_results.shape[0]} filas × {cv_results.shape[1]} columnas")
print(f"Cutoffs evaluados: {list(cv_results['cutoff'].drop_duplicates())}")

display(cv_results.head())
display(cv_evaluation)

print("\nMejor modelo promedio por métrica en CV:")
print_best_models(cv_evaluation, cv_models)

Resultados CV: 156 filas × 8 columnas
Cutoffs evaluados: [Timestamp('2021-03-29 00:00:00'), Timestamp('2022-03-28 00:00:00'), Timestamp('2023-03-27 00:00:00')]


,unique_id,ds,cutoff,y,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,2021-04-05,2021-03-29,60.0,69.0,209.0,78.0,69.097104
1,dengue_cali,2021-04-12,2021-03-29,77.0,69.0,177.0,78.0,69.194208
2,dengue_cali,2021-04-19,2021-03-29,51.0,69.0,161.0,78.0,69.291312
3,dengue_cali,2021-04-26,2021-03-29,46.0,69.0,142.0,78.0,69.388416
4,dengue_cali,2021-05-03,2021-03-29,39.0,69.0,155.0,78.0,69.485520


,unique_id,cutoff,metric,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,2021-03-29,mae,15.769231,39.192308,22.884615,17.873804
1,dengue_cali,2022-03-28,mae,13.961538,22.730769,11.000000,14.853738
2,dengue_cali,2023-03-27,mae,154.669231,176.515385,159.938462,153.836992
3,dengue_cali,2021-03-29,rmse,18.828170,54.814021,25.780881,21.153887
4,dengue_cali,2022-03-28,rmse,15.675802,27.270723,12.656035,16.732381
5,dengue_cali,2023-03-27,rmse,199.605800,224.944571,206.144738,198.168276
6,dengue_cali,2021-03-29,smape,0.135143,0.237494,0.181336,0.149228
7,dengue_cali,2022-03-28,smape,0.176543,0.240309,0.145776,0.185040
8,dengue_cali,2023-03-27,smape,0.523267,0.661536,0.554802,0.517781



Mejor modelo promedio por métrica en CV:
Mejor modelo por métrica
   MAE: Naive              15.7692
  RMSE: Naive              18.8282
 SMAPE: Naive              0.1351


In [63]:
cv_best_model, _ = best_model_by_metric(
    evaluation=cv_evaluation,
    metric="mae",
    models=cv_models,
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_model["ds"],
        y=df_model["y"],
        mode="lines",
        name="Serie real",
        line=dict(width=1.3),
    )
)

for index, cutoff_cv in enumerate(sorted(cv_results["cutoff"].unique())):
    fold = cv_results[cv_results["cutoff"] == cutoff_cv]

    fig.add_trace(
        go.Scatter(
            x=fold["ds"],
            y=fold[cv_best_model],
            mode="lines+markers",
            name=f"{cv_best_model} | fold {index + 1}",
            marker=dict(size=3),
            line=dict(width=2, dash="dot"),
        )
    )

    fig.add_shape(
        type="line",
        xref="x",
        yref="paper",
        x0=cutoff_cv,
        x1=cutoff_cv,
        y0=0,
        y1=1,
        line=dict(dash="dash", width=1),
        opacity=0.4,
    )

fig.update_layout(
    title=f"Validación cruzada temporal — Pronósticos de {cv_best_model}",
    xaxis_title="Fecha",
    yaxis_title="Casos",
    height=480,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)

fig.show()

---

## 17. Conclusión metodológica

Este flujo deja una secuencia más sólida para el análisis de series temporales:

1. Se construyó la serie semanal en formato Nixtla.
2. Se revisó la calidad de datos antes de modelar.
3. Se hizo el split temporal antes de imputar para evitar fuga de información.
4. Se entrenaron modelos baseline en escala original.
5. Se evaluaron los errores en el conjunto de prueba.
6. Se diagnosticaron los residuos del mejor baseline.
7. Se estudiaron autocorrelación, estacionariedad y transformaciones.
8. Se complementó la evaluación con validación cruzada temporal.

La lectura clave es que los baselines sirven como punto de comparación inicial.  
Si los residuos presentan autocorrelación o si las pruebas sugieren no estacionariedad, queda justificado avanzar hacia modelos más estructurados, como SARIMA, ETS, Prophet, modelos con regresores o enfoques de machine learning para series temporales.